# 01.2 · NumPy: indexing, boolean masks, broadcasting, random numbers

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/01-working-with-data/02-numpy-indexing-broadcasting/lesson.ipynb)

Read the [README](README.md) first. Here you check the by-hand examples, break broadcasting on purpose to read the error, then answer marketing's, pricing's and finance's requests on real data.

## 1. Indexing and slicing

In [1]:
import numpy as np
rev = np.array([15.30, 20.34, 22.00, 20.34, 20.34])

print("rev[0]   :", rev[0])
print("rev[-1]  :", rev[-1])
print("rev[1:3] :", rev[1:3])      # positions 1 and 2; 3 is excluded
print("rev[::2] :", rev[::2])      # every other element
print("rev[-2:] :", rev[-2:])      # last two

rev[0]   : 15.3
rev[-1]  : 20.34
rev[1:3] : [20.34 22.  ]
rev[::2] : [15.3  22.   20.34]
rev[-2:] : [20.34 20.34]


In [2]:
grid = np.arange(12).reshape(3, 4)
print(grid)
print("grid[1, 2] :", grid[1, 2])      # row 1, column 2
print("grid[0]    :", grid[0])         # whole row 0
print("grid[:, 0] :", grid[:, 0])      # whole column 0
print("grid[1:, 2:]:\n", grid[1:, 2:]) # bottom-right block

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
grid[1, 2] : 6
grid[0]    : [0 1 2 3]
grid[:, 0] : [0 4 8]
grid[1:, 2:]:
 [[ 6  7]
 [10 11]]


### A slice is a view

This is the trap from the README. Watch the original change.

In [3]:
first_three = rev[:3]
first_three[0] = 0            # change the slice...
print("rev is now:", rev)     # ...and the original changed too

rev = np.array([15.30, 20.34, 22.00, 20.34, 20.34])   # restore
safe = rev[:3].copy()
safe[0] = 0
print("rev after editing a copy:", rev)

rev is now: [ 0.   20.34 22.   20.34 20.34]
rev after editing a copy: [15.3  20.34 22.   20.34 20.34]


## 2. Boolean masks: select by condition

In [4]:
mask = rev > 20
print("mask       :", mask)
print("rev[mask]  :", rev[mask])
print("count      :", mask.sum())        # True counts as 1
print("fraction   :", mask.mean())       # 4 of 5

mask       : [False  True  True  True  True]
rev[mask]  : [20.34 22.   20.34 20.34]
count      : 4
fraction   : 0.8


Combine conditions with `&` (and), `|` (or), `~` (not). Each condition goes in brackets, because `&` binds tighter than `>`.

In [5]:
print("between 16 and 22 :", rev[(rev > 16) & (rev < 22)])
print("below 16 or above 21:", rev[(rev < 16) | (rev > 21)])
print("not above 20       :", rev[~(rev > 20)])

between 16 and 22 : [20.34 20.34 20.34]
below 16 or above 21: [15.3 22. ]
not above 20       : [15.3]


In [6]:
try:
    rev[(rev > 16) and (rev < 22)]
except ValueError as e:
    print("ValueError:", e)

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()


`and` asks "is this whole array true?", which has no single answer. Always `&`.

### Fancy indexing: pick by a list of positions

In [7]:
print(rev[[0, 4, 2]])          # positions 0, 4, 2, in that order
print(rev[np.argsort(rev)])    # argsort gives the positions that would sort the array
print(np.argsort(rev)[-2:])    # positions of the two largest

[15.3  20.34 22.  ]
[15.3  20.34 20.34 20.34 22.  ]
[4 2]


## 3. Broadcasting

Three product prices, four discount tiers. We want a 3 × 4 table.

In [8]:
prices = np.array([2.50, 3.40, 2.80])
tiers = np.array([1.0, 0.9, 0.8, 0.7])
print("prices shape:", prices.shape, " tiers shape:", tiers.shape)

try:
    prices * tiers
except ValueError as e:
    print("ValueError:", e)

prices shape: (3,)  tiers shape: (4,)
ValueError: operands could not be broadcast together with shapes (3,) (4,) 


`(3,)` against `(4,)`: compare from the right, 3 vs 4, neither is 1, so NumPy refuses. Turn prices into a **column** of shape `(3, 1)`; then `(3, 1)` against `(4,)` → `(3, 1)` vs `(1, 4)` → 1 vs 4 ✓, 3 vs 1 ✓.

In [9]:
col = prices[:, None]          # add an axis: shape (3, 1)
print("column shape:", col.shape)
table = col * tiers
print(table.round(2))
print("table shape:", table.shape, "   (by hand: first row 2.50, 2.25, 2.00, 1.75)")

column shape: (3, 1)
[[2.5  2.25 2.   1.75]
 [3.4  3.06 2.72 2.38]
 [2.8  2.52 2.24 1.96]]
table shape: (3, 4)    (by hand: first row 2.50, 2.25, 2.00, 1.75)


In [10]:
# Scalars and rows broadcast too
print((table * 1.2).shape)            # VAT on everything: () vs (3, 4)
print((table - table.mean(axis=0)).shape)   # subtract each column's mean: (3,4) - (4,)
print((table - table.mean(axis=1)[:, None]).shape)   # subtract each row's mean: (3,4) - (3,1)

(3, 4)
(3, 4)
(3, 4)


That last pattern, *subtract the column mean from every row*, is exactly what feature scaling does in Module 04. You already know how it works.

## 4. Random numbers with a seed

In [11]:
rng = np.random.default_rng(42)
print("dice      :", rng.integers(1, 7, size=5))
print("uniform   :", rng.random(3).round(3))
print("normal    :", rng.normal(loc=100, scale=15, size=3).round(1))   # mean 100, sd 15
print("choice    :", rng.choice(["UK", "DE", "FR"], size=5))

dice      : [1 5 4 3 3]
uniform   : [0.697 0.094 0.976]
normal    : [101.9  95.3  99.7]
choice    : ['FR' 'FR' 'DE' 'DE' 'DE']


In [12]:
a = np.random.default_rng(42).integers(1, 7, size=5)
b = np.random.default_rng(42).integers(1, 7, size=5)
c = np.random.default_rng(7).integers(1, 7, size=5)
print(a, b, "same seed → same numbers:", np.array_equal(a, b))
print(c,    "different seed → different numbers")

[1 5 4 3 3] [1 5 4 3 3] same seed → same numbers: True
[6 4 5 6 4] different seed → different numbers


Every train/test split, every model initialisation and every simulation in this course takes a seed, so that you and the solutions notebook get the same answer.

## 5. On real data

Marketing: order lines worth more than £100. Then UK only. Then the ten biggest.

In [13]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import sys, pandas as pd
sys.path.insert(0, str(data_path("download.py").parent))
from download import ensure_online_retail

retail = pd.read_excel(ensure_online_retail(), sheet_name="Year 2010-2011")
qty_real = retail["Quantity"].to_numpy()
price_real = retail["Price"].to_numpy()
country = retail["Country"].to_numpy()
revenue_real = qty_real * price_real
print(revenue_real.shape)

(541910,)


In [14]:
big = revenue_real > 100
print(f"lines over £100          : {big.sum():,}  ({big.mean():.2%} of all lines)")

uk_big = big & (country == "United Kingdom")
print(f"...of which UK           : {uk_big.sum():,}")
print(f"value of UK lines > £100 : £{revenue_real[uk_big].sum():,.0f}")

lines over £100          : 13,554  (2.50% of all lines)
...of which UK           : 10,373
value of UK lines > £100 : £3,113,930


In [15]:
top10 = np.argsort(revenue_real)[-10:][::-1]          # positions of the 10 largest, biggest first
for pos in top10:
    print(f"row {pos:>6}  qty {qty_real[pos]:>6}  price £{price_real[pos]:>8.2f}  revenue £{revenue_real[pos]:>10,.2f}  {country[pos]}")

row 540421  qty  80995  price £    2.08  revenue £168,469.60  United Kingdom
row  61619  qty  74215  price £    1.04  revenue £ 77,183.60  United Kingdom
row 222671  qty     60  price £  649.50  revenue £ 38,970.00  United Kingdom
row  15017  qty      1  price £13541.33  revenue £ 13,541.33  United Kingdom
row 299982  qty      1  price £11062.06  revenue £ 11,062.06  United Kingdom
row 173382  qty      1  price £ 8142.75  revenue £  8,142.75  United Kingdom
row 348325  qty   1412  price £    5.06  revenue £  7,144.72  United Kingdom
row  52711  qty   3114  price £    2.10  revenue £  6,539.40  United Kingdom
row 160546  qty   3114  price £    2.10  revenue £  6,539.40  United Kingdom
row 421601  qty   2400  price £    2.08  revenue £  4,992.00  Netherlands


Finance: simulate 1,000 possible monthly revenues. We take the real daily totals from 2011, draw 30 of them at random (with replacement) and add them up; repeat 1,000 times. This is the **bootstrap** idea, which returns properly in Module 05.

In [16]:
daily = retail.assign(rev=revenue_real).groupby(retail["InvoiceDate"].dt.date)["rev"].sum().to_numpy()
print(f"{daily.size} trading days, mean daily revenue £{daily.mean():,.0f}")

rng = np.random.default_rng(0)
sims = np.array([rng.choice(daily, size=30, replace=True).sum() for _ in range(1000)])
lo, mid, hi = np.percentile(sims, [5, 50, 95])
print(f"simulated 30-day revenue: 5th pct £{lo:,.0f}   median £{mid:,.0f}   95th pct £{hi:,.0f}")

305 trading days, mean daily revenue £31,960
simulated 30-day revenue: 5th pct £802,833   median £953,416   95th pct £1,130,786


Change the seed and the three numbers move a little; keep it and they are identical every run. That is reproducibility.

## Quiz answers

1. The last two elements, as a view.
2. A mask selects individual elements wherever it is `True`; those elements no longer form a rectangle, so NumPy returns them as a flat 1-D copy.
3. Compare shapes from the right; each pair of dimensions must be equal or one of them must be 1; the result takes the larger of each pair.
4. `(5, 3)` with `(5,)`: 3 vs 5 → **no**. `(5, 3)` with `(3,)`: 3 vs 3 ✓, 5 vs (missing = 1) ✓ → **yes**, result `(5, 3)`. To use the `(5,)` one, reshape it to `(5, 1)`.
5. It guarantees the same sequence of numbers every time it is created with seed 0, on any machine. It does not guarantee the numbers are "random-looking" in any particular way, nor that a *different* generator or library will produce the same sequence.

**Next:** [01.3 Pandas: Series, DataFrames, reading files, inspecting data](../03-pandas-basics/)